In [ ]:
# === Cell 1: Setup & Model (scoring + temporal smoothness regularization) ===
import os, re, random, warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import precision_recall_curve

# PyG
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, GATConv

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------ Reproducibility ------------------
def set_seed(seed: int = 452):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(4412)
os.makedirs("embedding", exist_ok=True)

# ------------------ IO & Data ------------------
@dataclass
class IOConfig:
    base_dir: str = "."
    cluster_ids: List[int] = (4, 8, 7, 3, 2, 1, 5)
    network_score_threshold: float = 0.1
    gold_file: Optional[str] = "chipseq_TF_TG_filtered.tsv"

def clean_suffix(name: str) -> str:
    return re.sub(r"_cluster\d+$", "", str(name))

def read_cluster_files(cfg: IOConfig, cid: int):
    p = cfg.base_dir
    tf_file   = os.path.join(p, f"cluster{cid}_allregulators.txt")
    gene_file = os.path.join(p, f"cluster{cid}_allGenes.txt")
    expr_file = os.path.join(p, f"cluster{cid}.table")
    net_file  = os.path.join(p, f"cluster{cid}_network.txt")

    assert os.path.exists(tf_file), f"Missing {tf_file}"
    assert os.path.exists(gene_file), f"Missing {gene_file}"
    assert os.path.exists(expr_file), f"Missing {expr_file}"
    assert os.path.exists(net_file), f"Missing {net_file}"

    tfs   = pd.read_csv(tf_file, header=None)[0].map(clean_suffix).tolist()
    genes = pd.read_csv(gene_file, header=None)[0].map(clean_suffix).tolist()

    expr = pd.read_csv(expr_file, sep="\t", index_col=0)
    expr.index = expr.index.map(clean_suffix)

    net = pd.read_csv(net_file, sep="\t", header=None)
    if net.shape[1] == 3:
        net.columns = ["TF", "TG", "score"]
    else:
        cols = ["TF", "TG"] + [f"col{i}" for i in range(2, net.shape[1])]
        net.columns = cols
        if "score" not in net.columns:
            net["score"] = 1.0
    net["TF"] = net["TF"].map(clean_suffix)
    net["TG"] = net["TG"].map(clean_suffix)
    net = net[net["score"] > cfg.network_score_threshold].copy()

    return tfs, genes, expr, net

def build_global_index(cfg: IOConfig):
    all_tfs, all_tgs = set(), set()
    for cid in cfg.cluster_ids:
        tfs, genes, _, _ = read_cluster_files(cfg, cid)
        all_tfs.update(tfs)
        all_tgs.update(genes)
    TF_list = sorted(all_tfs)
    TG_list = sorted(all_tgs)
    tf2idx = {t: i for i, t in enumerate(TF_list)}
    tg2idx = {g: i for i, g in enumerate(TG_list)}
    return TF_list, TG_list, tf2idx, tg2idx

def zscore_rows(mat: np.ndarray, eps=1e-8) -> np.ndarray:
    m = mat.mean(axis=1, keepdims=True)
    s = mat.std(axis=1, keepdims=True) + eps
    return (mat - m) / s

def pad_to(arr: np.ndarray, target_cols: int) -> np.ndarray:
    if arr.shape[1] >= target_cols:
        return arr
    pad = np.zeros((arr.shape[0], target_cols - arr.shape[1]), dtype=arr.dtype)
    return np.concatenate([arr, pad], axis=1)

def build_hetero_per_cluster(cfg: IOConfig,
                             TF_list: List[str],
                             TG_list: List[str],
                             tf2idx: Dict[str, int],
                             tg2idx: Dict[str, int]):
    cluster_data = {}
    max_cells = 0

    tmp_feats, tmp_edges = {}, {}
    inv_tf = TF_list
    inv_tg = TG_list

    for cid in cfg.cluster_ids:
        tfs, genes, expr, net = read_cluster_files(cfg, cid)
        tf_idx_sorted = sorted([tf2idx[x] for x in tfs if x in tf2idx])
        tg_idx_sorted = sorted([tg2idx[x] for x in genes if x in tg2idx])

        tf_names_sorted = [inv_tf[i] for i in tf_idx_sorted]
        tg_names_sorted = [inv_tg[i] for i in tg_idx_sorted]

        tf_expr = expr.reindex(tf_names_sorted).fillna(0.0).to_numpy(dtype=np.float32)
        tg_expr = expr.reindex(tg_names_sorted).fillna(0.0).to_numpy(dtype=np.float32)
        tf_expr = zscore_rows(np.log1p(tf_expr))
        tg_expr = zscore_rows(np.log1p(tg_expr))

        tmp_feats[cid] = (tf_idx_sorted, tf_expr, tg_idx_sorted, tg_expr)
        max_cells = max(max_cells, tf_expr.shape[1], tg_expr.shape[1])

        edges = []
        for _, row in net.iterrows():
            if row["TF"] in tf2idx and row["TG"] in tg2idx:
                edges.append((tf2idx[row["TF"]], tg2idx[row["TG"]]))
        tmp_edges[cid] = edges

    for cid in cfg.cluster_ids:
        tf_idx_sorted, tf_expr, tg_idx_sorted, tg_expr = tmp_feats[cid]
        tf_expr = pad_to(tf_expr.astype(np.float32), max_cells)
        tg_expr = pad_to(tg_expr.astype(np.float32), max_cells)

        data = HeteroData()
        num_TF_total = len(tf2idx)
        num_TG_total = len(tg2idx)
        TF_feat = np.zeros((num_TF_total, tf_expr.shape[1]), dtype=np.float32)
        TG_feat = np.zeros((num_TG_total, tg_expr.shape[1]), dtype=np.float32)
        TF_feat[tf_idx_sorted, :] = tf_expr
        TG_feat[tg_idx_sorted, :] = tg_expr

        data["TF"].x = torch.tensor(TF_feat, dtype=torch.float32)
        data["TG"].x = torch.tensor(TG_feat, dtype=torch.float32)

        edges = np.array(tmp_edges[cid], dtype=np.int64).T if len(tmp_edges[cid]) else np.zeros((2,0), dtype=np.int64)
        data["TF", "regulate", "TG"].edge_index = torch.tensor(edges, dtype=torch.long)
        rev = edges[::-1].copy() if edges.shape[1] > 0 else edges
        data["TG", "regulated_by", "TF"].edge_index = torch.tensor(rev, dtype=torch.long)

        cluster_data[cid] = data.to(DEVICE)

    return cluster_data, max_cells

# ------------------ Lineage (Topology) ------------------
def lineage_topology(selected_ids: List[int]):
    parent_full = {8:4, 7:8, 3:7, 2:3, 1:8, 5:1}
    topo_full   = [4, 8, 7, 3, 2, 1, 5]
    topo = [c for c in topo_full if c in selected_ids]
    parent = {c: (parent_full.get(c, None) if parent_full.get(c, None) in selected_ids else None) for c in topo}
    return topo, parent  # parent: cid -> parent_cid or None

# ------------------ Model ------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, GATConv
from torch_geometric.data import HeteroData
from typing import Dict, List, Optional, Tuple

# ===== PairNorm: keeps node embeddings from collapsing =====
class PairNorm(nn.Module):
    """
    Anti-over-smoothing:
    Keep the variance of the node embeddings in a reasonable range,
    so that the nodes do not collapse to one point.
    """
    def __init__(self, scale: float = 1.0):
        super().__init__()
        self.scale = scale

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (num_nodes, dim)
        mean = x.mean(dim=0, keepdim=True)          # mean over the graph
        x = x - mean
        # mean L2 norm over the nodes
        norm = x.pow(2).sum(dim=1, keepdim=True).sqrt().mean()
        return self.scale * x / (norm + 1e-6)


# ===== HeteroDynamicGRN_TF_TG with residuals and PairNorm =====
class HeteroDynamicGRN_TF_TG(nn.Module):
    def __init__(self, in_channels: int, hidden: int, out: int, dropout: float = 0.5):
        """
        in_channels: input feature size (= max_cells, cells after padding)
        hidden:      GRU hidden size
        out:         output embedding size
        dropout:     dropout probability
        """
        super().__init__()
        self.hidden = hidden
        self.out = out
        self.dropout = nn.Dropout(dropout)

        heads = 2
        gat_out = hidden * heads  # output size of each GATConv layer

        # --- heterogeneous GAT spatial layers ---
        # add_self_loops=False: self-loops on the bipartite graph raise a ValueError
        self.conv1 = HeteroConv({
            ("TF", "regulate", "TG"): GATConv(
                in_channels, hidden, heads=heads, add_self_loops=False
            ),
            ("TG", "regulated_by", "TF"): GATConv(
                in_channels, hidden, heads=heads, add_self_loops=False
            ),
        }, aggr="sum")

        self.conv2 = HeteroConv({
            ("TF", "regulate", "TG"): GATConv(
                gat_out, hidden, heads=heads, add_self_loops=False
            ),
            ("TG", "regulated_by", "TF"): GATConv(
                gat_out, hidden, heads=heads, add_self_loops=False
            ),
        }, aggr="sum")

        # --- BatchNorm ---
        self.bn_tf1 = nn.BatchNorm1d(gat_out)
        self.bn_tg1 = nn.BatchNorm1d(gat_out)
        self.bn_tf2 = nn.BatchNorm1d(gat_out)
        self.bn_tg2 = nn.BatchNorm1d(gat_out)

        # --- PairNorm: keeps node embeddings from shrinking ---
        self.pn_tf = PairNorm(scale=1.0)
        self.pn_tg = PairNorm(scale=1.0)

        # --- spatial residual: adds the projected input expression ---
        self.res_tf = nn.Linear(in_channels, gat_out)
        self.res_tg = nn.Linear(in_channels, gat_out)

        # --- temporal GRU ---
        self.gru_tf = nn.GRUCell(gat_out, hidden)
        self.gru_tg = nn.GRUCell(gat_out, hidden)

        # temporal residual: adds the linearly transformed spatial features to the GRU hidden state
        self.gru_skip_tf = nn.Linear(gat_out, hidden)
        self.gru_skip_tg = nn.Linear(gat_out, hidden)

        # --- output layer: hidden -> out ---
        self.fc_tf = nn.Linear(hidden, out)
        self.fc_tg = nn.Linear(hidden, out)

    # ===== spatial part (one time point) =====
    def spatial(self, data: HeteroData) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Input: the HeteroData of one cluster
        Output: the spatial TF / TG node embeddings of the cluster
               (num_TF, gat_out), (num_TG, gat_out)
        """
        # input features, for the residual
        x_tf0 = data["TF"].x  # (num_TF, in_channels)
        x_tg0 = data["TG"].x  # (num_TG, in_channels)

        x_dict = {"TF": x_tf0, "TG": x_tg0}
        edge_index_dict = {
            ("TF", "regulate", "TG"): data["TF", "regulate", "TG"].edge_index,
            ("TG", "regulated_by", "TF"): data["TG", "regulated_by", "TF"].edge_index,
        }

        # ---- GAT layer 1 ----
        h_dict = self.conv1(x_dict, edge_index_dict)
        h_tf = self.bn_tf1(h_dict["TF"])
        h_tg = self.bn_tg1(h_dict["TG"])
        h_tf = self.dropout(h_tf)
        h_tg = self.dropout(h_tg)

        # ---- GAT layer 2 ----
        h_dict2 = self.conv2({"TF": h_tf, "TG": h_tg}, edge_index_dict)
        h_tf2 = self.bn_tf2(h_dict2["TF"])
        h_tg2 = self.bn_tg2(h_dict2["TG"])

        # ---- PairNorm: spreads the node distribution ----
        h_tf2 = self.pn_tf(h_tf2)
        h_tg2 = self.pn_tg(h_tg2)

        # ---- residual: adds the projected input expression ----
        # keeps part of the input expression in each node embedding
        h_tf2 = h_tf2 + self.res_tf(x_tf0)
        h_tg2 = h_tg2 + self.res_tg(x_tg0)

        # dropout
        h_tf2 = self.dropout(h_tf2)
        h_tg2 = self.dropout(h_tg2)

        return h_tf2, h_tg2

    # ===== full forward pass: spatial + temporal =====
    def forward(
        self,
        datas: List[Tuple[int, HeteroData]],
        topo: List[int],
        parent: Dict[int, Optional[int]],
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        datas: [(cid, HeteroData), ...] in topo order
        topo:  [cid0, cid1, ...] cluster IDs in topological order
        parent: {cid: parent_cid or None}
        Returns:
            out_tf: (num_TF, T, out)
            out_tg: (num_TG, T, out)
        """
        # 1) spatial encoding of each cluster
        tf_spatial: Dict[int, torch.Tensor] = {}
        tg_spatial: Dict[int, torch.Tensor] = {}
        for cid, data in datas:
            tf_spatial[cid], tg_spatial[cid] = self.spatial(data)

        # 2) temporal propagation along the lineage with the GRU (with residual)
        h_tf: Dict[int, torch.Tensor] = {}
        h_tg: Dict[int, torch.Tensor] = {}

        for cid in topo:
            x_tf = tf_spatial[cid]   # (num_TF, gat_out)
            x_tg = tg_spatial[cid]   # (num_TG, gat_out)

            if parent[cid] is None:
                # root: the hidden state starts at 0
                h_prev_tf = x_tf.new_zeros(x_tf.size(0), self.hidden)
                h_prev_tg = x_tg.new_zeros(x_tg.size(0), self.hidden)
            else:
                h_prev_tf = h_tf[parent[cid]]
                h_prev_tg = h_tg[parent[cid]]

            # GRU update + residual:
            #   h_new = GRU(x, h_prev) + W_skip * x
            h_tf[cid] = self.gru_tf(x_tf, h_prev_tf) + self.gru_skip_tf(x_tf)
            h_tg[cid] = self.gru_tg(x_tg, h_prev_tg) + self.gru_skip_tg(x_tg)

        # 3) stack in topo order to (num_nodes, T, hidden)
        num_tf = next(iter(tf_spatial.values())).size(0)
        num_tg = next(iter(tg_spatial.values())).size(0)
        T = len(topo)

        tf_list = [h_tf[cid] for cid in topo]  # list[T] of (num_tf, hidden)
        tg_list = [h_tg[cid] for cid in topo]  # list[T] of (num_tg, hidden)

        tf_stack = torch.stack(tf_list, dim=1)  # (num_tf, T, hidden)
        tg_stack = torch.stack(tg_list, dim=1)  # (num_tg, T, hidden)

        # 4) linear map to the out size (per time step)
        tf_out = self.fc_tf(tf_stack)  # (num_tf, T, out)
        tg_out = self.fc_tg(tg_stack)  # (num_tg, T, out)

        return tf_out, tg_out


# ------------------ Scoring: absolute cosine similarity ------------------
USE_NORM   = False    # not used by score_probs
SCALE_ALPHA= 1.0      # not used by score_probs

def score_probs(out_tf: torch.Tensor,
                out_tg: torch.Tensor,
                eps: float = 1e-6):
    """
    Edge probability = absolute cosine similarity of the TF and TG embeddings.

    Input:
        out_tf: (num_TF, T, d)
        out_tg: (num_TG, T, d)

    Returns:
        sims:         list[np.ndarray], one (num_TF, num_TG) matrix per time point, values in [0, 1]
        probs_per_t:  list[Tensor], the same as tensors on the current device, for the BCE / smoothness losses
    """
    num_tf, T, d = out_tf.shape
    num_tg = out_tg.shape[0]

    sims = []
    probs_per_t = []

    for t in range(T):
        # TF / TG embeddings at time t: (num_tf, d), (num_tg, d)
        a = out_tf[:, t, :]
        b = out_tg[:, t, :]

        # L2-normalize each node to get cosine similarities
        a_n = F.normalize(a, p=2, dim=-1, eps=1e-8)   # (num_tf, d)
        b_n = F.normalize(b, p=2, dim=-1, eps=1e-8)   # (num_tg, d)

        # cosine similarity matrix: [-1, 1]
        cos_sim = torch.matmul(a_n, b_n.t())          # (num_tf, num_tg)

        # probability = absolute cosine similarity, in [0, 1]
        prob = cos_sim.abs()

        # clamp slightly inside (0, 1) so that BCE never takes log(0)
        prob = prob.clamp(min=eps, max=1.0 - eps)

        # store
        probs_per_t.append(prob)                       # for training / smoothness
        sims.append(prob.detach().cpu().numpy())       # for F1 / plots / reconstruction

    return sims, probs_per_t


# ------------------ Training / Utils ------------------
def build_labels_for_cluster(edges: np.ndarray, num_tf: int, num_tg: int, device) -> torch.Tensor:
    y = torch.zeros(num_tf * num_tg, dtype=torch.float32, device=device)
    if edges.size > 0:
        tf_idx = edges[0]
        tg_idx = edges[1]
        flat = torch.tensor(tf_idx * num_tg + tg_idx, dtype=torch.long, device=device)
        y[flat] = 1.0
    return y

def temporal_smoothness_loss(probs_per_t: List[torch.Tensor], topo: List[int], parent: Dict[int, Optional[int]]):
    """MSE(P_t, P_parent) averaged over the available parent-child pairs."""
    if len(probs_per_t) <= 1:
        return probs_per_t[0].new_tensor(0.0)
    cid2tid = {c:i for i,c in enumerate(topo)}
    smooth = probs_per_t[0].new_tensor(0.0)
    pairs = 0
    for c in topo:
        p = parent[c]
        if p is None: 
            continue
        i = cid2tid[c]; j = cid2tid[p]
        smooth = smooth + F.mse_loss(probs_per_t[i], probs_per_t[j])
        pairs += 1
    if pairs == 0:
        return probs_per_t[0].new_tensor(0.0)
    return smooth / pairs  # mean

def train_step(model, datas, topo, parent, optimizer, neg_ratio=1.0, lambda_smooth=1e-3):
    model.train()
    out_tf, out_tg = model(datas, topo, parent)
    sims, probs_per_t = score_probs(out_tf, out_tg)  # probabilities

    num_tf, num_tg = out_tf.shape[0], out_tg.shape[0]
    bce_total = out_tf.new_tensor(0.0)

    cid2data = {c: d for c, d in datas}
    for t, c in enumerate(topo):
        edges = cid2data[c]["TF", "regulate", "TG"].edge_index.detach().cpu().numpy()
        y = build_labels_for_cluster(edges, num_tf, num_tg, out_tf.device)
        probs = probs_per_t[t].reshape(-1)

        pos_idx = (y == 1).nonzero(as_tuple=False).view(-1)
        neg_idx = (y == 0).nonzero(as_tuple=False).view(-1)

        if len(pos_idx) > 0 and len(neg_idx) > 0:
            k = min(len(neg_idx), int(len(pos_idx) * neg_ratio))
            perm = torch.randperm(len(neg_idx), device=out_tf.device)[:k]
            idx = torch.cat([pos_idx, neg_idx[perm]], dim=0)
        else:
            idx = pos_idx

        if idx.numel() > 0:
            bce_total = bce_total + F.binary_cross_entropy(probs[idx], y[idx])

    # temporal smoothness regularization
    smooth = temporal_smoothness_loss(probs_per_t, topo, parent)
    total_loss = bce_total + (lambda_smooth * smooth)

    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    # returns three scalars: total loss / BCE / smoothness term (before λ)
    return float(total_loss.item()), float(bce_total.item()), float(smooth.item()), out_tf.detach(), out_tg.detach(), sims

def save_embeddings_txt(out_tf, out_tg, topo, subdir="embedding"):
    """
    Write the TF/TG embeddings of each cluster (cluster IDs from topo) as .txt:
    file names: embedding/tf_C{cid}.txt, embedding/tg_C{cid}.txt
    """
    import os, numpy as np, torch
    os.makedirs(subdir, exist_ok=True)
    tf = out_tf.detach().cpu().numpy() if isinstance(out_tf, torch.Tensor) else np.asarray(out_tf)
    tg = out_tg.detach().cpu().numpy() if isinstance(out_tg, torch.Tensor) else np.asarray(out_tg)
    # shapes: (num_TF, T, D), (num_TG, T, D)
    assert tf.shape[1] == len(topo) == tg.shape[1], "Embedding time dimension differs from the length of topo"
    for t, cid in enumerate(topo):
        np.savetxt(os.path.join(subdir, f"tf_C{cid}.txt"), tf[:, t, :], fmt="%.6f", delimiter="\t")
        np.savetxt(os.path.join(subdir, f"tg_C{cid}.txt"), tg[:, t, :], fmt="%.6f", delimiter="\t")

def compute_best_f1_from_sims(sims: List[np.ndarray],
                              gold_edges: List[Tuple[int,int]],
                              num_tg: int) -> float:
    best = 0.0
    pos_set = {(int(i), int(j)) for (i, j) in gold_edges}
    for S in sims:
        scores = S.reshape(-1)
        labels = np.zeros_like(scores, dtype=int)
        for (ti, gi) in pos_set:
            labels[ti * num_tg + gi] = 1
        p, r, _ = precision_recall_curve(labels, scores)
        f1 = np.where((p + r) > 0, 2 * p * r / (p + r), 0.0)
        best = max(best, float(f1.max() if len(f1) else 0.0))
    return best

def compute_smoothness_matrix(sims: List[np.ndarray]) -> Tuple[np.ndarray, float]:
    T = len(sims)
    M = np.zeros((T, T), dtype=float)
    for i in range(T):
        for j in range(T):
            M[i, j] = np.linalg.norm(sims[i] - sims[j])
    upper = M[np.triu_indices_from(M, k=1)]
    smoothness = 1.0 - float(np.mean(upper) / np.max(upper)) if len(upper) and np.max(upper) > 0 else 1.0
    return M, smoothness


In [ ]:
# === Cell 2 (Reproduce): Load checkpoint -> Forward -> Evaluate/Heatmap ===
# no training: load the model parameters from checkpoints/grn_best.pt and run the downstream steps

import os
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

# ====== 0) settings ======
CKPT_PATH = "checkpoint/grn_best.pt"
CMAP = "OrRd"

# device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert os.path.exists(CKPT_PATH), f"Checkpoint not found: {CKPT_PATH}"
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

meta = ckpt.get("meta", {}) if isinstance(ckpt, dict) else {}
state_dict = ckpt.get("model_state_dict", None) if isinstance(ckpt, dict) else None
assert state_dict is not None, "Checkpoint missing 'model_state_dict'."

# ====== 1) restore cfg, TF/TG lists, topo/parent and model hyperparameters from the checkpoint ======
meta_cfg = meta.get("cfg", None)
if isinstance(meta_cfg, dict):
    cfg = IOConfig(**meta_cfg)
elif meta_cfg is not None:
    cfg = meta_cfg
else:
    # settings used when the checkpoint holds no cfg
    cfg = IOConfig(
        base_dir=".",
        cluster_ids=[4, 8, 7, 3, 2, 1, 5],
        network_score_threshold=0.1,
        gold_file="chipseq_TF_TG_filtered.tsv"
    )

TF_list = meta.get("TF_list", None)
TG_list = meta.get("TG_list", None)
assert TF_list is not None and TG_list is not None, "Checkpoint missing TF_list/TG_list."

topo = meta.get("topo", None)
parent_cid = meta.get("parent_cid", None)
assert topo is not None and parent_cid is not None, "Checkpoint missing topo/parent_cid."

HIDDEN  = int(meta.get("HIDDEN", 64))
OUT     = int(meta.get("OUT", 64))
DROPOUT = float(meta.get("DROPOUT", 0.4))

max_cells_ckpt = int(meta.get("max_cells", -1))
assert max_cells_ckpt > 0, "Checkpoint missing/invalid max_cells."

# labels for printing and plots (topo order)
cid2tid = {c: i for i, c in enumerate(topo)}
cluster_labels = [f"C{c}" for c in topo]

# ====== 2) rebuild indices and data from the checkpoint's TF/TG lists (as in training) ======
tf2idx = {t: i for i, t in enumerate(TF_list)}
tg2idx = {g: i for i, g in enumerate(TG_list)}

cluster_data, max_cells_now = build_hetero_per_cluster(cfg, TF_list, TG_list, tf2idx, tg2idx)

# max_cells must match the first layer's input size, or the weights cannot be loaded
if int(max_cells_now) != int(max_cells_ckpt):
    raise ValueError(
        f"max_cells mismatch! built={max_cells_now}, ckpt={max_cells_ckpt}. "
        f"Use the same data and preprocessing as in training (cell numbers / table files)."
    )

datas = [(c, cluster_data[c]) for c in topo]

# ====== 3) rebuild the model and load the parameters (no training) ======
model = HeteroDynamicGRN_TF_TG(
    in_channels=max_cells_ckpt,
    hidden=HIDDEN,
    out=OUT,
    dropout=DROPOUT
).to(DEVICE)

model.load_state_dict(state_dict, strict=True)
model.eval()

# ====== 4) forward pass: out_tf/out_tg + sims/probs_per_t ======
# ====== 5) sims_best (from the checkpoint if stored, otherwise from the forward pass) ======
sims_best = ckpt.get("sims_best", None)

if sims_best is None:
    with torch.no_grad():
        out_tf_best, out_tg_best = model(datas, topo, parent_cid)
        sims_best, probs_per_t = score_probs(out_tf_best, out_tg_best)
else:
    sims_best = [np.asarray(S) for S in sims_best]
    probs_per_t = None  # not needed for the heatmaps / F1


# ====== 6) evaluation (best F1) ======
best_f1_value = None
gold_path = os.path.join(cfg.base_dir, cfg.gold_file) if cfg.gold_file else None
if gold_path and os.path.exists(gold_path):
    gold_df = pd.read_csv(gold_path, sep="\t")
    gTF, gTG = gold_df.columns[:2]
    gold_df[gTF] = gold_df[gTF].map(clean_suffix)
    gold_df[gTG] = gold_df[gTG].map(clean_suffix)

    gold_edges = [
        (tf2idx[t], tg2idx[g])
        for t, g in zip(gold_df[gTF], gold_df[gTG])
        if (t in tf2idx and g in tg2idx)
    ]
    best_f1_value = compute_best_f1_from_sims(sims_best, gold_edges, num_tg=len(tg2idx))
    print(f"Best F1: {best_f1_value:.4f}")
else:
    print("Best F1: (skipped: no gold-standard file found)")

# ====== 7) smoothness heatmaps (overall + each branch; cells show the Frobenius distance) ======
def plot_heatmap_with_labels(diffM: np.ndarray, labels: List[str], title: str, cmap: str = CMAP):
    plt.figure()
    plt.imshow(diffM, aspect="auto", cmap=cmap)
    plt.title(title)
    plt.colorbar(label="Frobenius norm diff")
    plt.xticks(ticks=np.arange(len(labels)), labels=labels, rotation=45, ha="right")
    plt.yticks(ticks=np.arange(len(labels)), labels=labels)

    n_rows, n_cols = diffM.shape
    for i in range(n_rows):
        for j in range(n_cols):
            plt.text(j, i, f"{diffM[i, j]:.2f}", ha="center", va="center", fontsize=12)

    plt.xlabel("Cluster")
    plt.ylabel("Cluster")
    plt.tight_layout()
    plt.show()

# overall heatmap
diffM_all, smoothness_all = compute_smoothness_matrix([np.asarray(S) for S in sims_best])
plot_heatmap_with_labels(
    diffM_all, cluster_labels,
    f"[All] Dissimilarity (smoothness={smoothness_all:.3f})",
    cmap=CMAP
)

# find all root→leaf branches and draw a heatmap for each
roots = [c for c in topo if parent_cid.get(c, None) is None]
root = roots[0] if roots else topo[0]

children_set = set(parent_cid.keys()) - {k for k, v in parent_cid.items() if v is None}
parent_set   = set(v for v in parent_cid.values() if v is not None)
leaves = [c for c in children_set if c not in parent_set]
if not leaves:
    leaves = [c for c in topo if c != root and c not in parent_set] or [root]

def path_to_root(node: int, parent_map: dict) -> list:
    path = [node]
    while parent_map.get(node, None) is not None:
        node = parent_map[node]
        path.append(node)
    return list(reversed(path))

branches = []
for leaf in leaves:
    b = [c for c in path_to_root(leaf, parent_cid) if c in cid2tid]
    if len(b) >= 2:
        branches.append(b)

# deduplicate: keep the longest unique paths
uniq, seen = [], set()
for b in sorted(branches, key=len, reverse=True):
    key = tuple(b)
    if key in seen:
        continue
    if not any(b != u and set(b).issubset(set(u)) for u in uniq):
        uniq.append(b)
        seen.add(key)
branches = uniq if uniq else branches

for b in branches:
    tids = [cid2tid[c] for c in b]
    sims_branch = [np.asarray(sims_best[t]) for t in tids]
    diffM_b, smooth_b = compute_smoothness_matrix(sims_branch)
    labels_b = [f"C{c}" for c in b]
    title = f"[Branch {'→'.join(labels_b)}] Dissimilarity (smoothness={smooth_b:.3f})"
    plot_heatmap_with_labels(diffM_b, labels_b, title, cmap=CMAP)

# ====== 8) training-loss curves (skipped when the checkpoint holds no loss history) ======
# plotted when the checkpoint holds loss_total_hist/loss_bce_hist/loss_smooth_hist;
# otherwise skipped.
print("Done. (No training performed; loaded model from checkpoint and evaluated.)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from typing import List

# ================= 1. global font settings =================
plt.rcParams.update({
    "axes.titlesize": 24,   # title
    "axes.labelsize": 20,   # axis labels
    "xtick.labelsize": 20,  # x ticks
    "ytick.labelsize": 20,  # y ticks
    "font.size": 20         # base size (legends, text)
})
# ===================================================

# heatmap drawing function
def plot_simplified_heatmap(
    diffM: np.ndarray, 
    labels: List[str], 
    title: str, 
    branches_texts: List[str], 
    cmap: str = "OrRd" # colormap
):
    # ================= normalize the matrix (divide by the maximum) =================
    max_val = np.nanmax(diffM)
    if max_val > 0:
        diffM = diffM / max_val

    fig, ax = plt.subplots(figsize=(11, 10))
    im = ax.imshow(diffM, aspect="auto", cmap=cmap)
    ax.set_title(title, pad=20)
    
    # larger colorbar font
    cbar = fig.colorbar(im, ax=ax)
    
    ax.set_xticks(ticks=np.arange(len(labels)))
    ax.set_yticks(ticks=np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    
    # annotate each cell with its value
    n_rows, n_cols = diffM.shape
    threshold = np.nanmax(diffM) / 2.0  
    for i in range(n_rows):
        for j in range(n_cols):
            text_color = "white" if diffM[i, j] > threshold else "black"
            ax.text(
                j, i, f"{diffM[i, j]:.2f}",
                ha="center", va="center",
                fontsize=20, color=text_color
            )

    # ================= blue dashed box at C4-C5 only (inset) =================
    delta = 0.06  # inset; larger values give a smaller box
    if "C4" in labels and "C5" in labels:
        idx_c4 = labels.index("C4")
        idx_c5 = labels.index("C5")
        
        # box at C4 (row) - C5 (column)
        rect1 = patches.Rectangle(
            (idx_c5 - 0.5 + delta, idx_c4 - 0.5 + delta), 1 - 2*delta, 1 - 2*delta,
            linewidth=4, edgecolor='blue', linestyle='--', facecolor='none', zorder=10
        )
        ax.add_patch(rect1)
        
        # symmetric box at C5 (row) - C4 (column)
        rect2 = patches.Rectangle(
            (idx_c4 - 0.5 + delta, idx_c5 - 0.5 + delta), 1 - 2*delta, 1 - 2*delta,
            linewidth=4, edgecolor='blue', linestyle='--', facecolor='none', zorder=10
        )
        ax.add_patch(rect2)

    # ================= smoothness of each branch below the plot =================
    # bottom margin from the text length (room for the rotated labels and 24 pt text)
    bottom_margin = 0.18 + 0.06 * len(branches_texts)
    plt.subplots_adjust(bottom=bottom_margin)

    if branches_texts:
        # offset keeps the text below the rotated tick labels
        text_y_start = bottom_margin - 0.10 
        for b_idx, text in enumerate(branches_texts):
            # regular weight, size 24, line spacing 0.06
            plt.figtext(
                0.48, text_y_start - (b_idx * 0.06), text, 
                ha="center", va="top", fontsize=22, color="black", weight="normal"
            )

    plt.show()


# ================= computation =================

# 1. overall heatmap matrix
diffM_all, smoothness_all = compute_smoothness_matrix([np.asarray(S) for S in sims_best])

# 2. find all root→leaf branches
roots = [c for c in topo if parent_cid[c] is None]
root = roots[0] if roots else topo[0]
children_set = set(parent_cid.keys()) - {k for k, v in parent_cid.items() if v is None}
parent_set   = set(v for v in parent_cid.values() if v is not None)
leaves = [c for c in children_set if c not in parent_set]
if not leaves:
    leaves = [c for c in topo if c != root and c not in parent_set] or [root]

def path_to_root(node: int, parent_map: dict) -> list:
    path = [node]
    while parent_map.get(node, None) is not None:
        node = parent_map[node]
        path.append(node)
    return list(reversed(path))

branches = []
for leaf in leaves:
    b = [c for c in path_to_root(leaf, parent_cid) if c in cid2tid]
    if len(b) >= 2:
        branches.append(b)

# deduplicate: keep the longest unique paths
uniq, seen = [], set()
for b in sorted(branches, key=len, reverse=True):
    key = tuple(b)
    if key in seen:
        continue
    if not any(b != u and set(b).issubset(set(u)) for u in uniq):
        uniq.append(b)
        seen.add(key)
branches = uniq if uniq else branches

# 3. per-branch text
all_branches_texts = []

for b in branches:
    labels_b = [f"C{c}" for c in b]
    
    # smoothness of this branch
    tids = [cid2tid[c] for c in b]
    sims_branch = [np.asarray(sims_best[t]) for t in tids]
    _, smooth_b = compute_smoothness_matrix(sims_branch)
    
    # text shown below the plot
    all_branches_texts.append(f"Branch {' → '.join(labels_b)}  |  Smoothness: {smooth_b:.3f}")

# 4. draw once
# colormap
title_all = f"[DRAFT] Frobenius distance (overall smoothness={smoothness_all:.3f})"
plot_simplified_heatmap(
    diffM=diffM_all, 
    labels=cluster_labels, 
    title=title_all, 
    branches_texts=all_branches_texts,
    cmap=CMAP # CMAP defined above
)

In [ ]:
# === Network export: global top-K + specificity-weighted TF rescue + final edge score >= 0.85 ===
import os, re
import numpy as np
import pandas as pd

# 0) check the required inputs
if "sims_best" not in globals() or sims_best is None or len(sims_best) == 0:
    raise NameError("sims_best not found.")
if "topo" not in globals() or topo is None or len(topo) == 0:
    raise NameError("topo not found.")
if "cfg" not in globals() or (not hasattr(cfg, "base_dir")):
    raise NameError("cfg.base_dir not found.")

S0 = np.asarray(sims_best[0])
n_tf, n_tg = S0.shape
print("[Info] sims_best[0] shape:", (n_tf, n_tg))

# 1) find the TF/TG lists
def _pick_list(candidates, target_len, kind):
    found = []
    for name in candidates:
        if name in globals() and isinstance(globals()[name], (list, tuple)):
            found.append((name, list(globals()[name])))
    for name, lst in found:
        if len(lst) == target_len:
            print(f"[OK] {kind}_LIST_USE <- {name} (len={len(lst)})")
            return lst
    msg = f"No list matches the {kind} dimension of sims_best (target_len={target_len}).\n"
    msg += "Candidate list lengths:\n"
    for name, lst in found:
        msg += f"  - {name}: {len(lst)}\n"
    raise ValueError(msg)

TF_LIST_USE = _pick_list(["TF_list_filtered", "TF_list", "TF_list_filter", "tf_list", "tfs"], n_tf, "TF")
TG_LIST_USE = _pick_list(["TG_list", "tg_list", "gene_list", "genes"], n_tg, "TG")

# 2) mappings
tf2idx = {tf: i for i, tf in enumerate(TF_LIST_USE)}
tg2idx = {tg: i for i, tg in enumerate(TG_LIST_USE)}
idx2tf = {i: tf for tf, i in tf2idx.items()}
idx2tg = {i: tg for tg, i in tg2idx.items()}

# 3) clean_suffix fallback
if "clean_suffix" not in globals() or (not callable(clean_suffix)):
    def clean_suffix(x: str) -> str:
        x = str(x)
        x = re.sub(r"([_\-\.]cluster\d+)$", "", x, flags=re.IGNORECASE)
        return x
    print("[Info] clean_suffix not defined; using the fallback.")

# 4) parameters
NETWORK_DIR = "network"
os.makedirs(NETWORK_DIR, exist_ok=True)

TOP_K_GLOBAL = 50000
ACTIVE_FRAC  = 0.3

M_MIN  = 0
M_MAX  = 5000
GAMMA  = 10.0

# kept edges must score >= 0.85
MIN_SCORE_KEEP = 0.85

EPS = 1e-12

# 5) active TFs/TGs
def get_active_indices_for_cluster(cid, tf2idx, tg2idx, TF_list, TG_list, cfg, active_frac=0.05):
    table_path = os.path.join(cfg.base_dir, f"cluster{cid}.table")
    if not os.path.exists(table_path):
        print(f"[Warn] Missing expression table for cluster {cid}: {table_path}")
        return [], []

    expr = pd.read_csv(table_path, sep="\t", index_col=0)
    expr.index = expr.index.map(clean_suffix)
    if expr.index.has_duplicates:
        expr = expr.groupby(expr.index).max()

    n_cells = expr.shape[1]
    if n_cells == 0:
        print(f"[Warn] Cluster C{cid}: expression has 0 cells.")
        return [], []

    thr = active_frac * n_cells
    expr_genes = set(expr.index)

    active_tf_idx = []
    for tf in TF_list:
        if tf in expr_genes and (expr.loc[tf].values > 0).sum() >= thr:
            active_tf_idx.append(tf2idx[tf])

    active_tg_idx = []
    for tg in TG_list:
        if tg in expr_genes and (expr.loc[tg].values > 0).sum() >= thr:
            active_tg_idx.append(tg2idx[tg])

    print(f"[Debug] C{cid}: active TF={len(active_tf_idx)}, active TG={len(active_tg_idx)}")
    return active_tf_idx, active_tg_idx

# 6) fraction of cells expressing each TF (>0)
def _load_cluster_expr(cid):
    table_path = os.path.join(cfg.base_dir, f"cluster{cid}.table")
    expr = pd.read_csv(table_path, sep="\t", index_col=0)
    expr.index = expr.index.map(clean_suffix)
    if expr.index.has_duplicates:
        expr = expr.groupby(expr.index).max()
    return expr

tf_frac_by_c = {}
for cid in topo:
    table_path = os.path.join(cfg.base_dir, f"cluster{cid}.table")
    if not os.path.exists(table_path):
        print(f"[Warn] missing {table_path}, skip in tf_frac_df")
        tf_frac_by_c[cid] = pd.Series(dtype=float)
        continue

    expr = _load_cluster_expr(cid)
    n_cells = expr.shape[1]
    if n_cells == 0:
        tf_frac_by_c[cid] = pd.Series(dtype=float)
        continue

    tfs_in = [tf for tf in TF_LIST_USE if tf in expr.index]
    if len(tfs_in) == 0:
        tf_frac_by_c[cid] = pd.Series(dtype=float)
        continue

    frac = (expr.loc[tfs_in].to_numpy() > 0).mean(axis=1)
    tf_frac_by_c[cid] = pd.Series(frac, index=tfs_in)

tf_frac_df = pd.DataFrame(tf_frac_by_c).fillna(0.0)
other_max_cache = {}
for cid in topo:
    cols_other = [c for c in tf_frac_df.columns if c != cid]
    other_max_cache[cid] = tf_frac_df[cols_other].max(axis=1) if len(cols_other) else pd.Series(0.0, index=tf_frac_df.index)

print("[OK] tf_frac_df built:", tf_frac_df.shape)

# 7) main loop
for t, cid in enumerate(topo):
    S = np.asarray(sims_best[t])
    if S.shape != (n_tf, n_tg):
        raise ValueError(f"sims_best[{t}] shape {S.shape} != expected {(n_tf, n_tg)}")

    active_tf_idx, active_tg_idx = get_active_indices_for_cluster(
        cid, tf2idx, tg2idx, TF_LIST_USE, TG_LIST_USE, cfg, active_frac=ACTIVE_FRAC
    )
    if len(active_tf_idx) == 0 or len(active_tg_idx) == 0:
        print(f"[Info] Cluster C{cid}: no active TF/TG (skip).")
        continue

    S_sub = S[np.ix_(active_tf_idx, active_tg_idx)]
    n_tf_active, n_tg_active = S_sub.shape
    if n_tf_active == 0 or n_tg_active == 0:
        print(f"[Info] Cluster C{cid}: empty S_sub (skip).")
        continue

    flat_scores = S_sub.ravel()
    n_edges = flat_scores.size
    if n_edges == 0:
        print(f"[Info] Cluster C{cid}: no candidate edges (skip).")
        continue

    # (A) global top-K among edges with score >= MIN_SCORE_KEEP
    eligible = np.where(flat_scores >= MIN_SCORE_KEEP)[0]
    if eligible.size == 0:
        print(f"[Info] Cluster C{cid}: no edges with score>={MIN_SCORE_KEEP} (skip).")
        continue

    K = min(TOP_K_GLOBAL, eligible.size)
    if K == eligible.size:
        top_idx = eligible[np.argsort(-flat_scores[eligible])]
    else:
        part = eligible[np.argpartition(-flat_scores[eligible], K - 1)[:K]]
        top_idx = part[np.argsort(-flat_scores[part])]

    edge_map = {}
    for idx in top_idx:
        i_local = idx // n_tg_active
        j_local = idx % n_tg_active
        tf_g = active_tf_idx[i_local]
        tg_g = active_tg_idx[j_local]
        s = float(flat_scores[idx])
        prev = edge_map.get((tf_g, tg_g))
        if prev is None or s > prev:
            edge_map[(tf_g, tg_g)] = s

    # (B) TF rescue, only edges with score >= MIN_SCORE_KEEP
    f_other_max = other_max_cache[cid]
    f_c_series  = tf_frac_df.get(cid, pd.Series(dtype=float))

    rescued_tf = 0
    for i_local, tf_g in enumerate(active_tf_idx):
        tf_name = idx2tf.get(tf_g, None)
        if tf_name is None:
            continue

        f_c  = float(f_c_series.get(tf_name, 0.0))
        f_om = float(f_other_max.get(tf_name, 0.0))
        spec = f_c / (f_c + f_om + EPS)

        M_tf = int(round(M_MIN + (M_MAX - M_MIN) * (spec ** GAMMA)))
        if M_tf <= 0:
            continue

        row = S_sub[i_local, :].astype(float)
        valid = np.where(row >= MIN_SCORE_KEEP)[0]   # same 0.85 cutoff
        if valid.size == 0:
            continue

        m = min(M_tf, valid.size)
        topm_local = valid[np.argpartition(-row[valid], m - 1)[:m]]
        topm_local = topm_local[np.argsort(-row[topm_local])]

        for j_local in topm_local:
            tg_g = active_tg_idx[int(j_local)]
            s = float(row[int(j_local)])
            prev = edge_map.get((tf_g, tg_g))
            if prev is None or s > prev:
                edge_map[(tf_g, tg_g)] = s

        rescued_tf += 1

    # (C) final output (filtered once more)
    top_edges = [(tf_g, tg_g, s) for (tf_g, tg_g), s in edge_map.items() if s >= MIN_SCORE_KEEP]
    top_edges.sort(key=lambda x: -x[2])

    out_path = os.path.join(NETWORK_DIR, f"C{cid}_network.txt")
    with open(out_path, "w") as f:
        f.write("TF\tTG\tScore\n")
        for tf_idx, tg_idx, score in top_edges:
            f.write(f"{idx2tf.get(tf_idx, tf_idx)}\t{idx2tg.get(tg_idx, tg_idx)}\t{score:.6f}\n")

    print(f"[OK] Cluster C{cid}: saved {len(top_edges)} edges (eligible={eligible.size}, globalK={K}, rescued_TFs={rescued_tf}) -> {out_path}")